# `sim.bench`: a whole bench, simulated
**Host only.** The simulation never runs on the board, so this notebook has no hardware
section. Every cell runs in ordinary Python (CPython) on this computer.

# TODO:
- Add an MCP23017 to the example project once it has a driver and a fake (not yet).

Decided (2026-11-16): `SimBench` takes the same routing config as the real bench, and builds
through the real `I2CManager` (with a `bus_factory`), so everything from the table down runs
the board's code. Each part's fake comes from `devices.fake_class` (its device notebook), and
`parts` sets each fake's starting state, as plain values or signals.

# Python file
location: `src/micropython_i2c/sim/bench.py`

tests: `tests/test_sim_bench.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/sim/bench.py
"""``SimBench``: a whole I2C bench on the host, from the same routing config as the real one.

``SimBench(config, parts)`` makes a ``FakeBus`` per bus, attaches each mux's and device's fake
(``devices.fake_class``) at its address and path, and builds the muxes and drivers through the
real ``I2CManager``. Run it inside ``with SimClock():`` to use simulated time::

    with SimClock() as clock:
        bench = SimBench(CONFIG, parts={"temp": {"temp_c": ramp(20.0, 26.0, 0, 3600)}})
        bench.build()
        clock.advance(1800)
        bench["temp"].temperature()      # about 23 C, half way up the ramp

``parts[name]`` is a dict of keyword arguments for that part's fake, or a ready-made fake
object (for a part with no registered fake, or one the test built itself).

Host only.

Generated by ``dev_notebooks/sim_bench_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None.

## Internal
The routing table, the manager, the registry (drivers and fakes) and `FakeBus`.

In [ ]:
%%extract ../src/micropython_i2c/sim/bench.py -a
from micropython_i2c.devices import fake_class, registry
from micropython_i2c.manager import I2CManager
from micropython_i2c.routing import RoutingTable
from micropython_i2c.sim.bus import FakeBus

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_sim_bench.py
import pytest

from micropython_i2c.compat import sleep_ms
from micropython_i2c.devices.drivers.qwiic_relay_fake import FakeQwiicRelay
from micropython_i2c.routing import ConfigError
from micropython_i2c.sim.bench import SimBench
from micropython_i2c.sim.bus import FakeRegisterPart
from micropython_i2c.sim.clock import SimClock
from micropython_i2c.sim.signals import lag, noisy, ramp, sequence, sine

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

from micropython_i2c.compat import sleep_ms
from micropython_i2c.devices.drivers.qwiic_relay_fake import FakeQwiicRelay
from micropython_i2c.routing import ConfigError
from micropython_i2c.sim.bus import FakeRegisterPart
from micropython_i2c.sim.clock import SimClock
from micropython_i2c.sim.signals import lag, noisy, ramp, sequence, sine

# Intent
**Purpose:** write and test a whole multi-sensor project before (or without) the hardware: the
same routing config, the same manager, the same drivers, with fake parts whose quantities
follow scenarios in simulated time.

```mermaid
flowchart LR
    C[routing config] --> T[RoutingTable.validate]
    T --> B[SimBench]
    P["parts: values and signals"] --> B
    B -->|one per bus| F[FakeBus]
    B -->|fake_class| K["FakeTMP102, FakeSHTC3, ..."]
    K -->|attach at address + path| F
    B -->|bus_factory| M[I2CManager]
    M --> D["drivers: bench['temp'], ..."]
```

**What it proves:** the config is valid, every part answers where the config says, mux routing
works, and the application logic does what it should as the world changes (a thermostat holds
its band, an alarm fires when a door opens). **What it does not:** that the code fits in the
board's RAM, or anything electrical. The hardware run still has the last word.

## What the module defines

| Name | What it is |
|---|---|
| `SimBench` | fake buses + fakes from a config, built through `I2CManager`; `bench[name]` is a driver, `bench.fake(name)` its fake |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/sim/bench.py -a
__all__ = ["SimBench"]

## `SimBench`
The config is validated against the registry first, so a bad table fails here with every
problem listed, as it would on the bench. A fake whose behaviour depends on its own address
(`TAKES_ADDR`) gets the configured address. Without a `board`, each bus gets a 400 kHz cap
(what the real board allows); pass the real board name to use its entry.

In [ ]:
%%extract ../src/micropython_i2c/sim/bench.py -a
class SimBench:
    """A fake bench built from a routing config.

    Args:
        config: The routing config (JSON text or dict), as for ``RoutingTable.loads``.
        parts: ``{name: kwargs dict or fake object}`` for muxes and devices; a name left out
            gets its fake with default arguments.
        board: A board name or dict for the bus clocks; default 400 kHz on every bus.

    Attributes:
        table: The validated ``RoutingTable``.
        buses: ``{bus_id: FakeBus}``.
        fakes: ``{name: fake part}`` for every mux and device.
        manager: The ``I2CManager``; ``build()`` builds its muxes and drivers.

    Raises:
        ConfigError: If the config is invalid.
        KeyError: If a driver has no registered fake and ``parts`` gives none.
    """

    def __init__(self, config, parts=None, board=None):
        table = RoutingTable.loads(config)
        names = {e["driver"] for _, e in table.entries() if e.get("driver")}
        reg = registry(sorted(names))
        self.table = table.validate(reg)
        self.buses = {bus_id: FakeBus() for bus_id in table.buses}
        self.fakes = {}
        parts = parts or {}
        for name, entry in table.entries():
            part = self._fake_for(name, entry, parts.get(name))
            path = table.resolve_path(entry["path"])
            self.buses[entry["bus"]].attach(entry["address"], part, path)
            self.fakes[name] = part
        if board is None:
            cap = {"scl": None, "sda": None, "baud_max": 400_000}
            board = {"buses": {b: dict(cap) for b in table.buses}}
        self.manager = I2CManager(self.table, reg, board, bus_factory=self._bus)

    @staticmethod
    def _fake_for(name, entry, given):
        if given is not None and not isinstance(given, dict):
            return given  # a ready-made fake
        kwargs = dict(given or {})
        cls = fake_class(entry["driver"])
        if getattr(cls, "TAKES_ADDR", False):
            kwargs.setdefault("addr", entry["address"])
        return cls(**kwargs)

    def _bus(self, bus_id, cfg, freq):
        return self.buses[bus_id]

    def build(self):
        """Build the buses, muxes and drivers (each driver checks its part's identity)."""
        self.manager.build()
        return self

    def scan(self, verbose=True):
        """``scan_check`` on the fake buses: does every part answer where the config says."""
        if not self.manager.buses:
            self.manager.buses_build()
        return self.manager.scan(verbose)

    def fake(self, name):
        """The fake part behind config name ``name``."""
        return self.fakes[name]

    def __getitem__(self, name):
        """The driver (or mux) named ``name``. Call ``build()`` first."""
        return self.manager[name]

### validation/test
The bench as wired today (README "The bench", 2026-11-14), plus the relay on bus 0. One
config, used by the cells below and by the tests.

In [ ]:
%%extract ../tests/test_sim_bench.py -a
BENCH = {
    "buses": [0, 1],
    "muxes": {"mux": {"bus": 1, "address": "0x70", "driver": "TCA9548A"}},
    "devices": [
        {"name": "air", "bus": 0, "address": "0x70", "driver": "SHTC3"},
        {"name": "heater", "bus": 0, "address": "0x18", "driver": "QwiicRelay"},
        {"name": "door", "bus": 1, "address": "0x22", "driver": "TMAG5273", "path": [["mux", 0]]},
        {"name": "soil", "bus": 1, "address": "0x48", "driver": "TMP102", "path": [["mux", 2]]},
        {"name": "tof", "bus": 1, "address": "0x29", "driver": "VL53L1X", "path": [["mux", 7]]},
    ],
}

#### validation: nominal usage
Build it, scan it, read each sensor once.

In [ ]:
with SimClock() as clock:
    bench = SimBench(BENCH, parts={"soil": {"temp_c": 21.5}, "air": {"temp_c": 22.0, "rh": 55.0}})
    ok, _ = bench.scan()
    bench.build()
    print("soil:", bench["soil"].temperature(), "C | air:", bench["air"].measure())
    bench["heater"].on()
    print("heater fake closed:", bench.fake("heater").closed, "| simulated time used: {:.4f} s".format(clock.now_s()))

#### validation: anti-nominal usage

In [ ]:
extra = {"name": "air2", "bus": 1, "address": "0x70", "driver": "SHTC3"}
bad = dict(BENCH, devices=BENCH["devices"] + [extra])
try:
    SimBench(bad)  # a second SHTC3 on bus 1 answers at 0x70, the mux's own address
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format(err))
else:
    raise AssertionError("an SHTC3 at the mux address should have raised ConfigError")

In [ ]:
odd = {"buses": [0], "devices": [{"name": "x", "bus": 0, "address": "0x48", "driver": "TMP102"}]}
sb = SimBench(odd, parts={"x": FakeRegisterPart()})  # a ready-made object is used as-is
try:
    sb.build()  # a TMP102 driver on a plain register file: its checks fail like on a wrong part
except Exception as err:
    print("raised {}, as expected for the wrong part at 0x48:\n    {}".format(type(err).__name__, err))
else:
    print("built: this TMP102 driver does no identity check on construction")

#### unit

In [ ]:
%%extract ../tests/test_sim_bench.py -a --strip-ipytest
%%ipytest


def test_bench_builds_and_every_part_answers():
    with SimClock():
        bench = SimBench(BENCH)
        ok, results = bench.scan(verbose=False)
        assert ok and len(results) == 6
        bench.build()
        assert set(bench.manager.devices) == {"air", "heater", "door", "soil", "tof"}

In [ ]:
%%extract ../tests/test_sim_bench.py -a --strip-ipytest
%%ipytest


def test_parts_set_the_fakes_and_drivers_read_them():
    with SimClock():
        bench = SimBench(BENCH, parts={"soil": {"temp_c": 21.5}, "air": {"rh": 55.0}}).build()
        assert bench["soil"].temperature() == pytest.approx(21.5, abs=0.07)
        assert bench["air"].measure()[1] == pytest.approx(55.0, abs=0.01)
        bench["heater"].on()
        assert bench.fake("heater").closed

In [ ]:
%%extract ../tests/test_sim_bench.py -a --strip-ipytest
%%ipytest


def test_an_invalid_config_is_refused_with_every_problem():
    extra = {"name": "air2", "bus": 1, "address": "0x70", "driver": "SHTC3"}
    bad = dict(BENCH, devices=BENCH["devices"] + [extra])
    with pytest.raises(ConfigError, match="collision on bus 1 at 0x70"):
        SimBench(bad)

In [ ]:
%%extract ../tests/test_sim_bench.py -a --strip-ipytest
%%ipytest


def test_a_ready_made_fake_is_attached_as_given():
    part = FakeRegisterPart()
    one = {"name": "x", "bus": 0, "address": "0x48", "driver": "TMP102"}
    bench = SimBench({"buses": [0], "devices": [one]}, parts={"x": part})
    assert bench.fake("x") is part
    assert bench.buses[0].scan() == [0x48]

In [ ]:
%%extract ../tests/test_sim_bench.py -a --strip-ipytest
%%ipytest


def test_a_ramp_shows_up_through_the_whole_stack():
    with SimClock() as clock:
        bench = SimBench(BENCH, parts={"soil": {"temp_c": ramp(20.0, 26.0, 0, 3600)}}).build()
        clock.advance(1800)
        assert bench["soil"].temperature() == pytest.approx(23.0, abs=0.1)

## Example: a whole project in simulated time
A small greenhouse controller on the bench config above, the kind of program that would run on
the board. Every minute it reads the soil temperature (TMP102) and drives the heater relay to
hold 20-22 C, logs the air (SHTC3), watches a door magnet (TMAG5273, Z axis) and looks for a
visitor (VL53L1X). It uses only the drivers and `compat.sleep_ms`, so the same function runs on
the board unchanged.

The world it runs in:

| Part | Scenario |
|---|---|
| heater relay | a `FakeQwiicRelay` built here, so the soil signal can read its `closed` |
| soil (TMP102) | `lag` towards 30 C while the heater is on, 12 C while off, time constant 15 min |
| air (SHTC3) | a 4-hour `sine` around 18 C; humidity 55 %RH with `noisy` spread |
| door (TMAG5273) | 25 mT on Z while the door is shut; 0.5 mT while open, from minute 60 to 80 |
| visitor (VL53L1X) | `sequence`: nobody (3 m) until minute 90, walks up to 0.4 m by 95, leaves by 105 |

In [ ]:
%%extract ../tests/test_sim_bench.py -a
def greenhouse(bench, minutes):
    """The application: one pass a minute. Returns one log row per minute."""
    heater, soil, air, door, tof = (bench[n] for n in ("heater", "soil", "air", "door", "tof"))
    door.configure(channels="XYZ")
    log = []
    for minute in range(minutes):
        t_soil = soil.temperature()
        if t_soil < 20.0:
            heater.on()
        elif t_soil > 22.0:
            heater.off()
        t_air, rh = air.measure()
        door_open = abs(door.magnetic()[2]) < 5.0  # the door magnet holds Z near 25 mT
        visitor = tof.measure().distance_mm < 800
        log.append((minute, t_soil, heater.state, t_air, rh, door_open, visitor))
        sleep_ms(60_000)
    return log


def greenhouse_world():
    """The scenario: the parts dict for SimBench, and the heater's fake."""
    heater = FakeQwiicRelay()
    parts = {
        "heater": heater,
        "soil": {"temp_c": lag(lambda t: 30.0 if heater.closed else 12.0, 900, initial=16.0)},
        "air": {"temp_c": sine(18.0, 3.0, 4 * 3600), "rh": noisy(55.0, 1.0, seed=1)},
        "door": {"field_mT": (0.0, 0.0, lambda t: 0.5 if 3600 <= t < 4800 else 25.0)},
        "tof": {"distance_mm": sequence([(0, 3000), (5400, 3000), (5700, 400), (6000, 400),
                                         (6300, 3000)])},
    }  # fmt: skip
    return parts, heater

Notebook-only: run the project for three simulated hours, print every 15th minute and a summary,
and how much real time it took.

In [ ]:
import time

wall = time.monotonic()
with SimClock() as clock:
    parts, heater_part = greenhouse_world()
    bench = SimBench(BENCH, parts=parts).build()
    log = greenhouse(bench, 180)
print("3 simulated hours in {:.2f} s of real time".format(time.monotonic() - wall))
print("min  soil C  heater  air C   %RH   door   visitor")
for minute, t_soil, on, t_air, rh, door_open, visitor in log[::15]:
    print("{:3d}  {:6.2f}  {!s:6}  {:5.2f}  {:5.1f}  {:6} {}".format(
        minute, t_soil, on, t_air, rh, "open" if door_open else "shut", "yes" if visitor else "-"))  # fmt: skip
after_warmup = [row[1] for row in log if row[0] >= 30]
print("soil after warm-up: {:.2f} .. {:.2f} C".format(min(after_warmup), max(after_warmup)))
print("door open minutes:", [r[0] for r in log if r[5]][0], "to", [r[0] for r in log if r[5]][-1])
print("visitor minutes:", [r[0] for r in log if r[6]])

#### integration
The same run as a test: the thermostat keeps the soil within its band (plus the overshoot the
lag allows), and the alarms fire in the right minutes and only then.

In [ ]:
%%extract ../tests/test_sim_bench.py -a --strip-ipytest
%%ipytest


def test_greenhouse_holds_its_band_and_sees_the_door_and_the_visitor():
    with SimClock():
        parts, heater = greenhouse_world()
        log = greenhouse(SimBench(BENCH, parts=parts).build(), 180)
    soil = [row[1] for row in log if row[0] >= 30]
    assert 19.0 < min(soil) and max(soil) < 23.0
    assert [r[0] for r in log if r[5]] == list(range(60, 80))
    visitor = [r[0] for r in log if r[6]]
    assert visitor and 92 <= visitor[0] and visitor[-1] <= 104
    assert any(r[2] for r in log) and not all(r[2] for r in log)  # the heater cycles